# Chapter 4 — Memory

Hands-On LangChain. Code targets **LangChain 1.x**.

A chat model is **stateless** — on its own it remembers nothing between calls. This chapter shows how
to give a conversation memory, and the four classic strategies for deciding *how much* of the past to
keep: the full buffer, a sliding window, a token budget, and a running summary.

Companion notebook for Chapter 4. Requires a running Ollama service and `MODEL` in `.env`.

## 4.0 Setup

In [ ]:
from model_provider import get_chat_model
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage, trim_messages

llm = get_chat_model(temperature=0.0)

## 4.1 Why conversations need memory

The model itself does not remember anything. Each call to the API is independent. To see this, let's
tell it our name in one call, then ask for it in a separate call.

In [ ]:
llm.invoke("Hi, my name is Youssef").content

In [ ]:
# A brand-new, independent call — the model never saw the previous one
llm.invoke("What is my name?").content

It cannot know. Chatbots appear to remember only because the application **resends the previous
turns** as context on every call. "Memory" is exactly that: storing the conversation so far and feeding
it back in. Let's build that.

## 4.2 Buffer memory: remember the whole conversation

We build a chain whose prompt has a `MessagesPlaceholder` for the running history, then wrap it with
`RunnableWithMessageHistory`. That wrapper automatically loads the stored messages, injects them into
the placeholder, runs the model, and saves the new turn back — all keyed by a `session_id`.

In [ ]:
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a friendly assistant. Answer concisely."),
    MessagesPlaceholder("history"),
    ("human", "{input}"),
])
chain = prompt | llm

# one history object per session id
store = {}
def get_session_history(session_id):
    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()
    return store[session_id]

conversation = RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="input",
    history_messages_key="history",
)

In [ ]:
config = {"configurable": {"session_id": "youssef"}}

print(conversation.invoke({"input": "Hi, my name is Youssef"}, config=config).content)

In [ ]:
print(conversation.invoke({"input": "What is 1+1?"}, config=config).content)

In [ ]:
# Thanks to memory, it now knows the answer
print(conversation.invoke({"input": "What is my name?"}, config=config).content)

Everything said so far is stored in that session's history:

In [ ]:
for m in store["youssef"].messages:
    print(f"{m.type:>5}: {m.content}")

This is **buffer memory**: keep the entire conversation. It is simplest and most faithful, but it
grows without bound — and since models charge per token, a long conversation gets slow and expensive.
The next strategies bound how much history we keep.

## 4.3 Window memory: keep only the last *k* turns

A sliding window keeps just the most recent messages and drops older ones. `trim_messages` with
`token_counter=len` counts *messages* (not tokens), so `max_tokens=2` keeps the last two messages —
one exchange.

In [ ]:
messages = [
    HumanMessage("Hi, my name is Youssef"), AIMessage("Hello Youssef!"),
    HumanMessage("What is 1+1?"),           AIMessage("1 + 1 equals 2."),
    HumanMessage("What is my name?"),        AIMessage("Your name is Youssef."),
]

windowed = trim_messages(
    messages, strategy="last", token_counter=len, max_tokens=2,
    start_on="human", include_system=False,
)
for m in windowed:
    print(f"{m.type:>5}: {m.content}")

Only the last exchange survives. To use this in a live chat, trim the history *before* it reaches
the prompt. Here the assistant keeps a window of 2 messages, so once the introduction scrolls out of the
window it can no longer recall the name.

In [ ]:
from langchain_core.runnables import RunnablePassthrough

def keep_last_k(messages, k=2):
    return trim_messages(messages, strategy="last", token_counter=len,
                         max_tokens=k, start_on="human", include_system=False)

windowed_chain = (
    RunnablePassthrough.assign(history=lambda x: keep_last_k(x["history"], k=2))
    | prompt | llm
)

store2 = {}
def get_history2(session_id):
    if session_id not in store2:
        store2[session_id] = InMemoryChatMessageHistory()
    return store2[session_id]

windowed_conv = RunnableWithMessageHistory(
    windowed_chain, get_history2,
    input_messages_key="input", history_messages_key="history",
)
cfg = {"configurable": {"session_id": "w"}}

print(windowed_conv.invoke({"input": "Hi, my name is Youssef"}, config=cfg).content)
print(windowed_conv.invoke({"input": "What is 1+1?"}, config=cfg).content)
print(windowed_conv.invoke({"input": "What is my name?"}, config=cfg).content)

## 4.4 Token-buffer memory: limit by tokens

Because pricing is per token, it is often more natural to cap the history by **token count** than by
number of messages. Use `trim_messages` with a real token counter. We'll use
`count_tokens_approximately` and watch the same short conversation shrink as we lower the budget.

In [ ]:
from langchain_core.messages.utils import count_tokens_approximately

convo = [
    HumanMessage("AI is what?!"),            AIMessage("Amazing!"),
    HumanMessage("Backpropagation is what?"), AIMessage("Beautiful!"),
    HumanMessage("Chatbots are what?"),       AIMessage("Charming!"),
]

def trim_to_tokens(messages, max_tokens):
    return trim_messages(messages, strategy="last",
                         token_counter=count_tokens_approximately,
                         max_tokens=max_tokens, start_on="human")

for limit in (100, 30, 10):
    kept = trim_to_tokens(convo, limit)
    print(f"max_tokens={limit:>3}: " + " | ".join(f"{m.type}:{m.content}" for m in kept))

As the budget drops, older exchanges are chopped off first, keeping the most recent turns within the limit.

## 4.5 Summary memory: compress older turns

Instead of dropping old messages, we can ask the LLM to **summarize** them, keeping the gist while
spending far fewer tokens. We keep the last couple of messages verbatim and replace everything older
with a one-paragraph summary stored as a system message.

In [ ]:
schedule = (
    "There is a meeting at 8am with your product team. You will need your powerpoint "
    "presentation prepared. 9am-12pm have time to work on your LangChain project which will "
    "go quickly because LangChain is such a powerful tool. At noon, lunch at the Italian "
    "restaurant with a customer who is driving over an hour to meet you to understand the "
    "latest in AI. Be sure to bring your laptop to show the latest LLM demo."
)

history = [
    HumanMessage("Hello"),                       AIMessage("What's up"),
    HumanMessage("Not much, just hanging"),       AIMessage("Cool"),
    HumanMessage("What is on the schedule today?"), AIMessage(schedule),
]

def summarize_history(messages, keep_last=2):
    if len(messages) <= keep_last:
        return messages
    older, recent = messages[:-keep_last], messages[-keep_last:]
    transcript = "\n".join(f"{m.type}: {m.content}" for m in older)
    summary = llm.invoke(f"Summarize this conversation concisely:\n\n{transcript}").content
    return [SystemMessage(f"Summary of earlier conversation: {summary}")] + recent

compressed = summarize_history(history, keep_last=2)
for m in compressed:
    print(f"{m.type:>6}: {m.content}\n")

The early small talk is collapsed into a compact summary, while the most recent turn is preserved
verbatim. Feeding `compressed` back into the prompt gives the model the context it needs at a fraction
of the token cost.

> For production apps that need durable, multi-session memory, **LangGraph** (Part III) provides built-in
> persistence via checkpointers. The strategies here — buffer, window, token, summary — are exactly the
> knobs you tune on top of it.